# Rossmann Supply Chain Demand Forecasting

### Notebook 1 - Data Cleaning

In [1]:
# 1. IMPORTS

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)


In [2]:
# 2. LOAD RAW FILES

train  = pd.read_csv("train.csv",        dtype={'StateHoliday': str}, low_memory=False)
store  = pd.read_csv("store.csv")
states = pd.read_csv("store_states.csv")

print("Train shape :", train.shape)
print("Store shape :", store.shape)
print("States shape:", states.shape)


Train shape : (1017209, 9)
Store shape : (1115, 10)
States shape: (1115, 2)


In [3]:
# 3. INITIAL INSPECTION

display(train.head())
display(store.head())
display(states.head())

print("\nMissing Values (Train):")
print(train.isnull().sum())

print("\nMissing Values (Store):")
print(store.isnull().sum())




,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


,Store,State
0,1,"HB,NI"
1,2,TH
2,3,NW
3,4,BE
4,5,SN



Missing Values (Train):
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

Missing Values (Store):
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


In [4]:
print("\nTrain Summary Statistics:")
display(train.describe())

print("\nStore Summary Statistics:")
display(store.describe())


Train Summary Statistics:


,Store,DayOfWeek,Sales,Customers,Open,Promo,SchoolHoliday
count,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06
mean,5.584297e+02,3.998341e+00,5.773819e+03,6.331459e+02,8.301067e-01,3.815145e-01,1.786467e-01
std,3.219087e+02,1.997391e+00,3.849926e+03,4.644117e+02,3.755392e-01,4.857586e-01,3.830564e-01
min,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.800000e+02,2.000000e+00,3.727000e+03,4.050000e+02,1.000000e+00,0.000000e+00,0.000000e+00
50%,5.580000e+02,4.000000e+00,5.744000e+03,6.090000e+02,1.000000e+00,0.000000e+00,0.000000e+00
75%,8.380000e+02,6.000000e+00,7.856000e+03,8.370000e+02,1.000000e+00,1.000000e+00,0.000000e+00
max,1.115000e+03,7.000000e+00,4.155100e+04,7.388000e+03,1.000000e+00,1.000000e+00,1.000000e+00



Store Summary Statistics:


,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear
count,1115.00000,1112.000000,761.000000,761.000000,1115.000000,571.000000,571.000000
mean,558.00000,5404.901079,7.224704,2008.668857,0.512108,23.595447,2011.763573
std,322.01708,7663.174720,3.212348,6.195983,0.500078,14.141984,1.674935
min,1.00000,20.000000,1.000000,1900.000000,0.000000,1.000000,2009.000000
25%,279.50000,717.500000,4.000000,2006.000000,0.000000,13.000000,2011.000000
50%,558.00000,2325.000000,8.000000,2010.000000,1.000000,22.000000,2012.000000
75%,836.50000,6882.500000,10.000000,2013.000000,1.000000,37.000000,2013.000000
max,1115.00000,75860.000000,12.000000,2015.000000,1.000000,50.000000,2015.000000


In [5]:
# 4. REMOVE INVALID ROWS
# Only open stores with positive sales

before = len(train)
train = train[train['Open'] == 1]
train = train[train['Sales'] > 0]
print(f"Removed {before - len(train):,} rows | Remaining: {len(train):,}")


Removed 172,871 rows | Remaining: 844,338


 Removed closed stores and zero sales to model actual demand only

In [6]:
# 5. CLEAN CATEGORICALS

train['StateHoliday'] = (
    train['StateHoliday']
    .astype(str)
    .str.strip()
    .replace({'0': 'none', 'a': 'public', 'b': 'easter', 'c': 'christmas'})
)

print("StateHoliday after fix:")
print(train['StateHoliday'].value_counts())


StateHoliday after fix:
StateHoliday
none         843428
public          694
easter          145
christmas        71
Name: count, dtype: int64


In [7]:
# 6. HANDLE MISSING VALUES

store['CompetitionDistance'] = store.groupby('StoreType')['CompetitionDistance']    .transform(lambda x: x.fillna(x.median()))

store['CompetitionOpenSinceMonth'] = store['CompetitionOpenSinceMonth'].fillna(0)
store['CompetitionOpenSinceYear']  = store['CompetitionOpenSinceYear'].fillna(0)
store['Promo2']                    = store['Promo2'].fillna(0)
store['Promo2SinceWeek']           = store['Promo2SinceWeek'].fillna(0)
store['Promo2SinceYear']           = store['Promo2SinceYear'].fillna(0)
store['PromoInterval']             = store['PromoInterval'].fillna('none')

print("Store nulls after fix:")
print(store.isnull().sum())


Store nulls after fix:
Store                        0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
dtype: int64


In [8]:
# 7. DATE FEATURES

train = train.drop(columns=['DayOfWeek'], errors='ignore')

train['Date']      = pd.to_datetime(train['Date'])
train['Year']      = train['Date'].dt.year
train['Month']     = train['Date'].dt.month
train['Week']      = train['Date'].dt.isocalendar().week.astype(int)
train['Day']       = train['Date'].dt.day
train['DayOfWeek'] = train['Date'].dt.dayofweek + 1   # Monday=1 … Sunday=7
train['IsWeekend'] = train['DayOfWeek'].isin([6, 7]).astype(int)   # FIX: was missing


In [9]:
# 8. MERGE DATA

df = train.merge(store,  on='Store', how='left')
df = df.merge(states, on='Store', how='left')
print("Merged shape:", df.shape)


Merged shape: (844338, 24)


In [10]:
# 9. VALIDATION CHECKS

print("Missing State values:", df['State'].isnull().sum())
print("Unique Stores       :", df['Store'].nunique())
print("Date Range          :", df['Date'].min(), "→", df['Date'].max())

print("\nNull counts across all columns:")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "  No nulls")


Missing State values: 0
Unique Stores       : 1115
Date Range          : 2013-01-01 00:00:00 → 2015-07-31 00:00:00

Null counts across all columns:
  No nulls


In [11]:
# 10. SORT DATA
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)


In [12]:
# Reduce skew in competition distance
df['CompetitionDistance'] = np.log1p(df['CompetitionDistance'])

# Ensure no duplicate Store-Date records
assert df.duplicated(['Store','Date']).sum() == 0

In [13]:
# FINAL CLEANED DATA SUMMARY

print("Final Cleaned Data:")
display(df.describe())

print("Final Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())

assert df.isnull().sum().sum() == 0

Final Cleaned Data:


,Store,Date,Sales,Customers,Open,Promo,SchoolHoliday,Year,Month,Week,Day,DayOfWeek,IsWeekend,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear
count,844338.000000,844338,844338.000000,844338.000000,844338.0,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000,844338.000000
mean,558.421374,2014-04-11 01:08:38.729702656,6955.959134,762.777166,1.0,0.446356,0.193578,2013.831945,5.845774,23.646946,15.835706,3.520350,0.174865,7.646005,4.926482,1369.692738,0.498670,11.596159,1003.201259
min,1.000000,2013-01-01 00:00:00,46.000000,8.000000,1.0,0.000000,0.000000,2013.000000,1.000000,1.000000,1.000000,1.000000,0.000000,3.044522,0.000000,0.000000,0.000000,0.000000,0.000000
25%,280.000000,2013-08-16 00:00:00,4859.000000,519.000000,1.0,0.000000,0.000000,2013.000000,3.000000,11.000000,8.000000,2.000000,0.000000,6.566672,0.000000,0.000000,0.000000,0.000000,0.000000
50%,558.000000,2014-03-31 00:00:00,6369.000000,676.000000,1.0,0.000000,0.000000,2014.000000,6.000000,23.000000,16.000000,3.000000,0.000000,7.754053,4.000000,2006.000000,0.000000,0.000000,0.000000
75%,837.000000,2014-12-11 00:00:00,8360.000000,893.000000,1.0,1.000000,0.000000,2014.000000,8.000000,35.000000,23.000000,5.000000,0.000000,8.836519,9.000000,2011.000000,1.000000,22.000000,2012.000000
max,1115.000000,2015-07-31 00:00:00,41551.000000,7388.000000,1.0,1.000000,1.000000,2015.000000,12.000000,52.000000,31.000000,7.000000,1.000000,11.236658,12.000000,2015.000000,1.000000,50.000000,2015.000000
std,321.730861,NaN,3103.815515,401.194153,0.0,0.497114,0.395102,0.777271,3.323959,14.389931,8.683392,1.723712,0.379852,1.558655,4.283634,935.556484,0.499999,15.308101,1005.874685


Final Shape: (844338, 24)
Missing values: 0


In [14]:
# 11. SAVE OUTPUT

df.to_csv("cleaned_rossmann.csv", index=False)
print("Saved: cleaned_rossmann.csv")
print("Final shape:", df.shape)


Saved: cleaned_rossmann.csv
Final shape: (844338, 24)
